In [ ]:
%%writefile inst.sh
#!/bin/bash
set -euo pipefail

cd /content

if [[ ! -d /content/foundry/.git ]]; then
  rm -rf /content/foundry
  git clone https://github.com/pubuyun/foundry.git /content/foundry
fi

cd /content/foundry

UV_BIN="$(command -v uv || true)"

if [[ -z "$UV_BIN" ]]; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
  UV_BIN="$(command -v uv || true)"
fi

if [[ -z "$UV_BIN" && -x "$HOME/.local/bin/uv" ]]; then
  UV_BIN="$HOME/.local/bin/uv"
fi

if [[ -z "$UV_BIN" && -x "/usr/local/bin/uv" ]]; then
  UV_BIN="/usr/local/bin/uv"
fi

if [[ -z "$UV_BIN" ]]; then
  echo "uv installation failed: uv binary not found." >&2
  exit 1
fi

echo "Using uv: $UV_BIN"

export UV_CACHE_DIR="/content/foundry/.uv-cache"
export UV_PYTHON_INSTALL_DIR="/content/foundry/.python"


"$UV_BIN" python install 3.12
"$UV_BIN" venv --clear --seed --python 3.12 .venv

.venv/bin/python -m pip install --upgrade pip setuptools wheel
.venv/bin/python -m pip install "rc-foundry[all]"

echo "Done."
echo "Python: /content/foundry/.venv/bin/python"
echo "Foundry: /content/foundry/.venv/bin/foundry"

Writing inst.sh


In [ ]:
!bash inst.sh

In [ ]:
!.venv/bin/foundry install base-models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CACHE = "/content/drive/MyDrive/foundry_cache/checkpoints"
DST = "/root/.foundry/checkpoints"

os.makedirs(DST, exist_ok=True)

# Restore full directory from Google Drive
!rsync -ah --info=progress2 "{CACHE}/" "{DST}/"

print("Loaded models from cache:", CACHE)
print("Restored to:", DST)

Mounted at /content/drive
          6.51G 100%   36.57MB/s    0:02:49 (xfr#5, to-chk=0/6)
Loaded models from cache: /content/drive/MyDrive/foundry_cache/checkpoints
Restored to: /root/.foundry/checkpoints


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

SRC = "/root/.foundry/checkpoints"
CACHE = "/content/drive/MyDrive/foundry_cache/checkpoints"

os.makedirs(os.path.dirname(CACHE), exist_ok=True)

# Copy full directory to Google Drive
!rsync -ah --info=progress2 "{SRC}/" "{CACHE}/"

print("Saved models cache to:", CACHE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
          2.69G  46%   96.65MB/s    0:00:26 (xfr#1, to-chk=0/5)
Saved models cache to: /content/drive/MyDrive/foundry_cache/checkpoints


In [ ]:
%cd /content/foundry
!git pull
!bash patch_rf3_t4_bf16.sh
# !.venv/bin/python models/rf3/src/rf3/inference.py inputs='tests/rf3input.json' out_dir="rf3out/"
!.venv/bin/rf3 fold inputs='tests/rf3input.json' out_dir="rf3out/"

/content/foundry
Already up to date.
already patched: foundry: /content/foundry/src/foundry/__init__.py
already patched: foundry: /content/foundry/.venv/lib/python3.12/site-packages/foundry/__init__.py
already patched: rf3 attention: /content/foundry/models/rf3/src/rf3/model/layers/attention.py
already patched: rf3 attention: /content/foundry/.venv/lib/python3.12/site-packages/rf3/model/layers/attention.py
RF3 Tesla T4 BF16/cuEquivariance patch installed.
06:07:30 DEBUG transforms: Debug mode is on
06:07:30 INFO rf3.inference_engines.rf3: [rank: 0] Loading checkpoint from /root/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...
06:07:33 WARNING atomworks.ml: Using element type for atom names of atomized tokens.
Using bfloat16 Automatic Mixed Precision (AMP)
06:07:39 INFO rf3.inference_engines.rf3: [rank: 0] Outputs will be written to /content/foundry/rf3out.
06:07:44 WARNING atomworks.io.utils.ccd: The following CCD codes were not found in the local mirror at : {np.str_('L: